# 📗 Module 04 · Notebook 05 — SLM Spesialis: Kecil tapi Cakap

"Kenapa banyak Small Language Model (SLM) dirilis, padahal katanya bodoh?" Justru karena di dunia
nyata **SLM bukan generalis serba-bisa — mereka pekerja spesialis yang efisien.**

## Kenapa SLM bukan bodoh?
1. **"Kecil" itu relatif.** SLM 2026 lebih pintar daripada model raksasa beberapa tahun lalu — berkat data & teknik, terutama **distillation** (model kecil belajar dari output model raksasa).
2. **Alat yang tepat.** Klasifikasi, ekstraksi, routing, redaksi PII, jawab-dari-konteks — **tak perlu 70B**.
3. **Spesialisasi > generalisasi.** SLM yang di-fine-tune untuk satu tugas bisa **mengalahkan model raksasa generalis** di tugas itu. ("Sempit tapi dalam.")
4. **Murah, cepat, privat.** 10-100× lebih murah/cepat; bisa jalan on-device (data tak keluar).
5. **Masa depan Agentic AI.** Riset NVIDIA: agen melakukan banyak sub-tugas sempit & berulang → ideal untuk SLM murah-cepat.

## Yang akan kita buktikan
Latih **Qwen3-0.6B** (mungil!) jadi spesialis **ekstraksi → JSON**, lalu tunjukkan ia
**mengalahkan generalis SmolLM3-3B** (5× lebih besar) di tugas itu — **lebih akurat DAN lebih
cepat** — dengan angka, bukan klaim.

In [1]:
# Qwen3 & SmolLM3 butuh transformers >=4.53; pin <5 (gotcha 4-bit & API lama)
!pip install -q "transformers>=4.53,<5" peft bitsandbytes accelerate datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.8 MB/s eta 0:00:00


In [2]:
import torch, json, re, time, random
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SPECIALIST = "Qwen/Qwen3-0.6B"           # mungil — akan kita fine-tune jadi spesialis
GENERALIST = "HuggingFaceTB/SmolLM3-3B"  # generalis pembanding (~5x lebih besar)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def gpu_mem(tag=""):
    if not torch.cuda.is_available(): return
    print(f"[{tag}] allocated={torch.cuda.memory_allocated()/1e9:.2f} GB | reserved={torch.cuda.memory_reserved()/1e9:.2f} GB")

device: cuda


## 1. Tugas: Ekstraksi Pesanan (+Diskon) → JSON

Ubah teks pesanan Indonesia berantakan menjadi JSON:
`{"produk":..., "jumlah":..., "harga_satuan":..., "diskon_persen":..., "total":...}`.

Sengaja sulit: model harus membaca angka gaya Indonesia ("25.000"), menangkap **diskon**, dan
**menghitung** `total = jumlah × harga_satuan × (100 − diskon) / 100`. Tugas seperti ini berat
untuk SLM mungil *zero-shot*, tapi mudah dipelajari lewat fine-tune — itulah "sempit tapi dalam".

In [3]:
random.seed(42)
PRODUK = ["minyak goreng","beras","gula pasir","kopi bubuk","teh celup","sabun mandi",
          "mie instan","tepung terigu","susu kental","garam"]
SATUAN = ["botol","kg","bungkus","kotak","sachet","pak"]
HARGA  = [5000,8500,12000,14000,17500,25000,30000,45000]
DISKON = [0,5,10,15,20,25]
TEMPLATES = [
    "Tolong kirim {j} {s} {p}, harga {hf} per {s}, diskon {d}%.",
    "Saya mau pesan {p} sebanyak {j} {s}, harganya {hf}, potongan {d} persen.",
    "Order: {p} {j} {s}, @{hf}, disc {d}%.",
    "Pesanan {j} {s} {p}, harga satuan {hf} rupiah, diskon {d} persen.",
    "minta {p} {j} {s} ya, {hf} satunya, diskon {d}%",
]
def gen_one():
    p=random.choice(PRODUK); s=random.choice(SATUAN)
    j=random.randint(1,20); h=random.choice(HARGA); d=random.choice(DISKON)
    hf=f"{h:,}".replace(",",".")                     # 25000 -> "25.000"
    text=random.choice(TEMPLATES).format(p=p,s=s,j=j,hf=hf,d=d)
    total=j*h*(100-d)//100                            # harga kelipatan 100 -> selalu bulat
    gold={"produk":p,"jumlah":j,"harga_satuan":h,"diskon_persen":d,"total":total}
    return {"text":text,"gold":gold}

data=[gen_one() for _ in range(250)]
train_data, test_data = data[:200], data[200:]
print(f"train={len(train_data)} | test={len(test_data)}")
print("contoh:", train_data[0]["text"], "->", json.dumps(train_data[0]["gold"], ensure_ascii=False))

train=200 | test=50
contoh: Saya mau pesan beras sebanyak 9 botol, harganya 14.000, potongan 5 persen. -> {"produk": "beras", "jumlah": 9, "harga_satuan": 14000, "diskon_persen": 5, "total": 119700}


## 2. Baseline Zero-shot (Belum Dilatih)

Ukur dua baseline pada teks uji yang sama:
- **Qwen3-0.6B base** (mungil, belum dilatih tugas ini)
- **SmolLM3-3B** (generalis, ~5× lebih besar)

Metrik: **valid-JSON rate** + **akurasi field** (5 field benar?) + **kecepatan** (token/detik).

In [4]:
SYS = ("Ekstrak detail pesanan menjadi JSON dengan key persis: produk, jumlah, harga_satuan, "
       "diskon_persen, total. Semua angka tanpa titik/koma. Keluarkan HANYA JSON, tanpa penjelasan.")

def extract(model, tok, text, max_new_tokens=64):
    messages=[{"role":"system","content":SYS},{"role":"user","content":text}]
    try:
        inputs=tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                       return_dict=True, enable_thinking=False).to(model.device)
    except TypeError:
        inputs=tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt",
                                       return_dict=True).to(model.device)
    out=model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                       pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def parse_json(s):
    m=re.search(r"\{.*\}", s, re.S)
    if not m: return None
    try: return json.loads(m.group())
    except Exception: return None

def evaluate(model, tok, dataset, label=""):
    valid=0; hits=0; total=0
    for ex in dataset:
        obj=parse_json(extract(model, tok, ex["text"])); g=ex["gold"]
        if obj is None:
            total+=len(g); continue
        valid+=1
        for k,v in g.items():
            total+=1
            if str(obj.get(k)).strip()==str(v).strip(): hits+=1
    res={"label":label,"valid":valid/len(dataset),"field_acc":hits/total}
    print(f"  [{label}] valid-JSON={res['valid']:.0%} | field-acc={res['field_acc']:.0%}")
    return res

def bench_batch_tps(model, tok, prompts, n_tokens=32):
    # Throughput produksi: proses BANYAK prompt sekaligus (batch). Kembalikan total token/detik.
    msgs=[[{"role":"system","content":SYS},{"role":"user","content":p}] for p in prompts]
    try:
        texts=[tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False,
                                       enable_thinking=False) for m in msgs]
    except TypeError:
        texts=[tok.apply_chat_template(m, add_generation_prompt=True, tokenize=False) for m in msgs]
    old=tok.padding_side; tok.padding_side="left"
    enc=tok(texts, return_tensors="pt", padding=True).to(model.device)
    tok.padding_side=old
    model.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)  # warm-up
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0=time.time()
    model.generate(**enc, min_new_tokens=n_tokens, max_new_tokens=n_tokens,
                   do_sample=False, pad_token_id=tok.eos_token_id)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    return len(prompts)*n_tokens/(time.time()-t0)

def load_4bit(name):
    bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    tok=AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl=AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto")
    return mdl, tok

In [5]:
# Baseline 1 — Qwen3-0.6B base (4-bit; nanti model ini yang kita fine-tune)
model, tok = load_4bit(SPECIALIST)
vram_spec = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
gpu_mem("Qwen3-0.6B base")
print("Contoh output base (zero-shot):")
print("  teks:", test_data[0]["text"])
print("  out :", extract(model, tok, test_data[0]["text"]))
r_base = evaluate(model, tok, test_data, "Qwen3-0.6B base")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[Qwen3-0.6B base] allocated=0.54 GB | reserved=1.32 GB
Contoh output base (zero-shot):
  teks: Order: minyak goreng 18 bungkus, @8.500, disc 15%.


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  out : {"produk": "minyak goreng", "jumlah": 18, "harga_satuan": 8500, "diskon_persen": 15, "total": 12750}
  [Qwen3-0.6B base] valid-JSON=100% | field-acc=78%


## 3. Baseline Generalis SmolLM3-3B

Generalis kuat, ~5× lebih besar. Muat 4-bit untuk inferensi, ukur akurasi + kecepatan, lalu
bebaskan memorinya sebelum training.

In [6]:
gen_model, gen_tok = load_4bit(GENERALIST)
vram_gen = (torch.cuda.memory_allocated()/1e9 - vram_spec) if torch.cuda.is_available() else 0
gpu_mem("SmolLM3-3B")
r_gen = evaluate(gen_model, gen_tok, test_data, "SmolLM3-3B generalis")
gen_btps = bench_batch_tps(gen_model, gen_tok, [d["text"] for d in test_data[:16]])
print(f"  throughput batch-16 SmolLM3-3B: {gen_btps:.0f} token/detik")
del gen_model, gen_tok
torch.cuda.empty_cache()
gpu_mem("setelah SmolLM3 dibebaskan")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[SmolLM3-3B] allocated=2.53 GB | reserved=4.08 GB
  [SmolLM3-3B generalis] valid-JSON=92% | field-acc=72%
  throughput batch-16 SmolLM3-3B: 101 token/detik
[setelah SmolLM3 dibebaskan] allocated=2.01 GB | reserved=4.08 GB


## 4. Latih Spesialis (QLoRA)

Ubah Qwen3-0.6B base jadi **spesialis** lewat QLoRA: format data dengan chat template (jawaban =
JSON gold), latih adapter kecil. Qwen3 hybrid-thinking; untuk tugas format ini `enable_thinking=
False` agar output langsung JSON.

In [7]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

def to_text(ex):
    messages=[{"role":"system","content":SYS},{"role":"user","content":ex["text"]},
              {"role":"assistant","content":json.dumps(ex["gold"], ensure_ascii=False)}]
    try:
        return {"text": tok.apply_chat_template(messages, tokenize=False, enable_thinking=False)}
    except TypeError:
        return {"text": tok.apply_chat_template(messages, tokenize=False)}

train_ds = Dataset.from_list(train_data).map(to_text)
def tokenize(ex): return tok(ex["text"], truncation=True, max_length=256)
tok_ds = train_ds.map(tokenize, remove_columns=train_ds.column_names)

model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args=TrainingArguments(output_dir="qwen3-extractor-lora", per_device_train_batch_size=4,
    gradient_accumulation_steps=2, num_train_epochs=8, learning_rate=2e-4, fp16=True,
    logging_steps=10, optim="paged_adamw_8bit", report_to="none", save_strategy="no")
Trainer(model=model, args=args, train_dataset=tok_ds,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)).train()
model.config.use_cache=True; model.eval()

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.362700
20,0.314400
30,0.185300
40,0.131200
50,0.120300
60,0.104500
70,0.103400
80,0.099600
90,0.097100
100,0.097000


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

## 5. Head-to-Head: Spesialis vs Generalis

Ukur spesialis 0.6B (fine-tuned) dan bandingkan dengan dua baseline.
- **Akurasi**: hipotesis — spesialis mungil mengalahkan generalis 5× lebih besar.
- **Efisiensi**: SLM menang di **memori** & **throughput saat batch** (kondisi produksi nyata).
  Catatan jujur: pada *satu* permintaan (batch-1) latensi keduanya mirip (overhead-bound) — bukan
  di situ keunggulan SLM, melainkan saat melayani banyak permintaan sekaligus.

In [8]:
r_spec = evaluate(model, tok, test_data, "Qwen3-0.6B SPESIALIS")
spec_btps = bench_batch_tps(model, tok, [d["text"] for d in test_data[:16]])

print("\n=== HEAD-TO-HEAD (akurasi) ===")
print(f"{'model':30}{'params':>8}{'valid-JSON':>12}{'field-acc':>11}")
for r,pp in [(r_base,"0.6B"),(r_gen,"3.0B"),(r_spec,"0.6B")]:
    print(f"{r['label']:30}{pp:>8}{r['valid']:>11.0%}{r['field_acc']:>11.0%}")

print("\n=== EFISIENSI (di sinilah SLM menang) ===")
print(f"  Memori   : Qwen3-0.6B ~{vram_spec:.2f} GB  vs  SmolLM3-3B ~{vram_gen:.2f} GB"
      f"  ->  {vram_gen/max(vram_spec,1e-6):.1f}x lebih hemat")
print(f"  Throughput batch-16 : SmolLM3-3B {gen_btps:.0f}  vs  Qwen3-0.6B {spec_btps:.0f} tok/dtk"
      f"  ->  {spec_btps/max(gen_btps,1e-6):.1f}x lebih tinggi")

print("\n--- contoh: base vs spesialis (model SAMA, beda adapter) ---")
ex=test_data[1]
with model.disable_adapter():
    base_out=extract(model, tok, ex["text"])
print("teks     :", ex["text"])
print("gold     :", json.dumps(ex["gold"], ensure_ascii=False))
print("base     :", base_out)
print("spesialis:", extract(model, tok, ex["text"]))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  [Qwen3-0.6B SPESIALIS] valid-JSON=100% | field-acc=95%

=== HEAD-TO-HEAD (akurasi) ===
model                           params  valid-JSON  field-acc
Qwen3-0.6B base                   0.6B       100%        78%
SmolLM3-3B generalis              3.0B        92%        72%
Qwen3-0.6B SPESIALIS              0.6B       100%        95%

=== EFISIENSI (di sinilah SLM menang) ===
  Memori   : Qwen3-0.6B ~0.54 GB  vs  SmolLM3-3B ~1.99 GB  ->  3.7x lebih hemat
  Throughput batch-16 : SmolLM3-3B 101  vs  Qwen3-0.6B 104 tok/dtk  ->  1.0x lebih tinggi

--- contoh: base vs spesialis (model SAMA, beda adapter) ---
teks     : Saya mau pesan teh celup sebanyak 13 sachet, harganya 25.000, potongan 0 persen.
gold     : {"produk": "teh celup", "jumlah": 13, "harga_satuan": 25000, "diskon_persen": 0, "total": 325000}
base     : {"produk": "teh celup", "jumlah": 13, "harga_satuan": 25000, "diskon_persen": 0, "total": 25000}
spesialis: {"diskon_persen": 0, "harga_satuan": 25000, "jumlah": 13, "produk": "te

## Ringkasan & Jembatan

| Pelajaran | Inti |
|-----------|------|
| SLM bukan bodoh | mereka **spesialis efisien**, bukan generalis serba-bisa |
| Spesialis > generalis | SLM **0.6B** yang di-fine-tune **mengalahkan generalis 3B** (5× lebih besar) di tugasnya |
| Efisien | jauh lebih **hemat memori** + **throughput batch lebih tinggi** (bukan latensi batch-1) |
| Resep | dataset tugas → QLoRA (notebook 03) → ukur jujur (notebook 04) |

Di **notebook 06** kita pakai resep yang sama untuk **tugas spesialis lain** (klasifikasi/intent
routing, sentimen/aspect, asisten domain) + benchmark ukuran 0.6B vs 1.7B vs 3B.